# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmed0607/ML-Internship-Strter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

# **Archetype to Action Mapping**

Based on our 4-cluster K-Means model on structural and engagement metrics:

  Cluster 0 — At-Risk / Declining Decay (rewrite or expand):

*  Profile: Mature pages, slipping positions, moderate-to-high historical visibility experiencing negative trend momentum.

*  Reason Code: at_risk_structural_decay

*  Action: Editorial refresh, updating stale data/facts, improving keyword relevance and snippet structure.

  Cluster 1 — Champions / Core Assets (protect & monitor):

*  Profile: High impressions, top average positions (1–5), strong engagement.

*  Reason Code: high_visibility_champion

*  Action: Protect core URL structure, monitor SERP fluctuations, avoid aggressive rewrites.

  Cluster 2 — Hidden Gems / Rising Potential (improve_internal_links):

*  Profile: Strong CTR and engagement metrics, but constrained by lower impression volume and page 2/3 rank.

*  Reason Code: high_intent_growth_opportunity

*  Action: Build internal linking from Champion pages and optimize title/meta tags for primary query capture.

  Cluster 3 — Underperforming / Thin Inventory (prune or merge):

*  Profile: Minimal impressions, low word count, low session capture over 90+ days.

*  Reason Code: thin_low_demand_candidate

*  Action: Audit for 301 redirection into broader topic guides, canonicalization, or pruning.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## **Intended Use**

This model acts strictly as a decision-support triage tool for editorial and SEO content teams. It ingests historical search performance and structural metadata to rank content pages into action queues so teams focus limited editorial capacity on high-impact opportunities first.

## **Known Limits**

*  Observational, Not Causal: The model identifies structural associations and historical momentum; it cannot guarantee that updating a page will causally reverse a ranking decline.

*  Semantic Blind Spot: Because raw text and search queries are excluded for privacy and safety, the model evaluates page structure and behavior, not semantic topical depth.

*  Platform Changes: SERP feature volatility (e.g., AI Overviews, knowledge panels) can suppress clicks without changing underlying content quality.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## **Human Review Protocol**

Before executing an action flagged by the queue, an editor must verify:

1.  Intent Integrity: Ensure the page is an informational/transactional content piece, not a static legal or navigational utility (e.g., Privacy Policy, Contact Us).

2.  Seasonality Check: Verify whether recent traffic drops match annual calendar cycles rather than genuine content obsolescence.

3.  Consolidation Impact: Check if a sibling page on the same domain has cannibalized or absorbed the query volume.

## **The No-Go List (What Should NOT Be Automated)**

*  Do NOT auto-delete or auto-redirect pages based on cluster scores alone without human confirmation.

*  Do NOT auto-generate mass content rewrites using generative AI without domain-expert fact-checking.

*  Do NOT alter URLs or canonical tags through automated scripts based on cluster membership.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## **Monitoring and Retraining Policy**

*  Cadence: Re-run cluster inference monthly on trailing 90-day aggregated performance snapshots.

*  Data Drift Triggers: Retrain scaler and cluster centroids if the median impressions per page shifts by more than 25% across active clients.

*  Cluster Stability Trigger: If more than 30% of inventory switches archetypes between consecutive monthly runs without underlying traffic changes, audit feature distributions for drift.

*  Concept Drift Triggers: Retrain whenever major search engine layout shifts or tracking definition updates occur in Search Console / GA4 sources.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import euclidean_distances


df = pd.read_csv('content_refresh_anonymized.csv')


df['avg_position'] = pd.to_numeric(df['avg_position'], errors='coerce').fillna(0)
df['word_count'] = pd.to_numeric(df['word_count'], errors='coerce')
df['ctr'] = pd.to_numeric(df['ctr'], errors='coerce').fillna(0)
df['impressions_90d'] = pd.to_numeric(df['impressions_90d'], errors='coerce').fillna(0)
df['sessions_90d'] = pd.to_numeric(df['sessions_90d'], errors='coerce').fillna(0)
df['content_age_days'] = pd.to_numeric(df['content_age_days'], errors='coerce').fillna(0)

df['has_avg_position'] = (df['avg_position'] > 0).astype(int)
median_pos = df[df['avg_position'] > 0]['avg_position'].median()
if pd.isna(median_pos): median_pos = 10.0
df['avg_position_clean'] = df['avg_position'].replace(0, median_pos)

df['has_word_count'] = df['word_count'].notnull().astype(int)
df['word_count_clean'] = df['word_count'].fillna(0)
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

features = ['impressions_90d', 'sessions_90d', 'content_age_days',
            'avg_position_clean', 'has_avg_position', 'word_count_clean',
            'has_word_count', 'ctr']

X = df[features].copy().astype(float)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

risk_cluster = df.groupby('cluster')['is_declining_label'].mean().idxmax()
distances = euclidean_distances(X_scaled, kmeans.cluster_centers_)
df['priority_score'] = (1.0 / (1.0 + distances[:, risk_cluster])) * 100.0

action_map = {
    risk_cluster: ('rewrite_or_expand', 'at_risk_structural_decay'),
}

def assign_action(c):
    if c == risk_cluster:
        return 'rewrite_or_expand', 'at_risk_structural_decay'
    return 'monitor_or_optimize', 'structural_opportunity'

df['suggested_action'], df['reason_code'] = zip(*df['cluster'].apply(assign_action))

os.makedirs('work/outputs', exist_ok=True)
ranked_queue = df.sort_values(by='priority_score', ascending=False)
output_cols = ['content_id', 'client_id', 'cluster', 'priority_score', 'suggested_action', 'reason_code', 'impressions_90d', 'avg_position_clean']
ranked_queue[output_cols].head(500).to_csv('work/outputs/model_action_playbook.csv', index=False)
print("Saved ranked queue to work/outputs/model_action_playbook.csv")

os.makedirs('work/figures', exist_ok=True)
plt.figure(figsize=(8, 5))
scatter = plt.scatter(np.log1p(df['impressions_90d']), df['avg_position_clean'], c=df['cluster'], cmap='viridis', alpha=0.3, s=10)
plt.xlabel('Log(1 + Impressions 90d)')
plt.ylabel('Cleaned Average Position')
plt.title('Content Inventory Archetype Clusters')
plt.gca().invert_yaxis()
plt.colorbar(scatter, label='Cluster ID')
plt.tight_layout()
plt.savefig('work/figures/archetype_clusters.png', dpi=200)
plt.close()
print("Saved cluster figure to work/figures/archetype_clusters.png")

print(f"\nTotal items processed: {len(df)}")
print(f"Top 5 Priority Review Items:\n{ranked_queue[['content_id', 'priority_score', 'suggested_action', 'reason_code']].head(5)}")

Saved ranked queue to work/outputs/model_action_playbook.csv
Saved cluster figure to work/figures/archetype_clusters.png

Total items processed: 30000
Top 5 Priority Review Items:
                 content_id  priority_score   suggested_action  \
28967  content_3b4febea02af       83.264136  rewrite_or_expand   
24909  content_4340e8d66f85       79.847647  rewrite_or_expand   
5327   content_fe16a55cd13d       78.866047  rewrite_or_expand   
18003  content_3360c3000d9f       78.752511  rewrite_or_expand   
12901  content_2665bb7de609       78.537018  rewrite_or_expand   

                    reason_code  
28967  at_risk_structural_decay  
24909  at_risk_structural_decay  
5327   at_risk_structural_decay  
18003  at_risk_structural_decay  
12901  at_risk_structural_decay  


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.